# TFT model


1. Imports
2. Load dataset
3. Clean dataset
4. Create timeseries structure
5. Create TFT dataset
6. Create model
7. Train model
8. Monitor training
9. Generate predictions
10. Signal detection
11. Save model

## Some boring shit before we get started
### Environment Setup for the TFT Notebook

The following steps describe how the Python environment used for this notebook was created.
This ensures reproducibility of the Temporal Fusion Transformer (TFT) experiments.

#### 1. Install Python 3.11

Install **Python 3.11 (64-bit)**.
Newer versions (e.g., 3.13) may cause compatibility problems with PyTorch and related libraries.

---

#### 2. Create a virtual environment

Open **PowerShell** in the project directory and run:

```
py -3.11 -m venv tft_env
```

---

#### 3. Activate the virtual environment

```
.\tft_env\Scripts\activate
```

---

#### 4. Upgrade pip

```
python -m pip install --upgrade pip
```

---

#### 5. Install the core libraries

First install PyTorch:

```
pip install torch torchvision torchaudio
```

Then install the remaining dependencies:

```
pip install pytorch-lightning pytorch-forecasting torchmetrics pandas numpy matplotlib scikit-learn jupyterlab
```

---

#### 6. Create a dedicated Jupyter kernel

Install the kernel package (**It's like creating a personal kernel for a specific enviroment, isolating it from eventual bugs from other directory files**):

```
pip install ipykernel
```

Register the environment as a Jupyter kernel:

```
python -m ipykernel install --user --name tft_env
```

---

#### 7. Start JupyterLab

```
jupyter lab
```

Inside JupyterLab, select the kernel from 'Kernel' upside:

```
tft_env
```

---

#### 8. Verify the installation

Run the following code to confirm that the environment is correctly configured:

```python
import torch
import pytorch_forecasting
import pytorch_lightning

print("torch:", torch.__version__)
print("forecasting:", pytorch_forecasting.__version__)
print("lightning:", pytorch_lightning.__version__)
print("GPU:", torch.cuda.is_available())
```

Expected output example:

```
torch: 2.11.0+cpu
forecasting: 1.6.1
lightning: 2.6.1
GPU: False
```

This confirms that the environment is properly configured and ready for the TFT pipeline.


# Now we start

## 1. Imports

In [1]:
!pip install pytorch-lightning pytorch-forecasting torchmetrics

  Using cached pytorch_lightning-2.6.1-py3-none-any.whl.metadata (21 kB)
  Using cached pytorch_forecasting-1.6.1-py3-none-any.whl.metadata (14 kB)
  Using cached torchmetrics-1.9.0-py3-none-any.whl.metadata (23 kB)
  Using cached torch-2.11.0-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached lightning_utilities-0.15.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached lightning-2.6.1-py3-none-any.whl.metadata (44 kB)
  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached scikit_base-0.13.1-py3-none-any.whl.metadata (8.8 kB)
  Using cached aiohttp-3.13.5-cp313-cp313-win_amd64.whl.metadata (8.4 kB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached aiohappyeyeballs-


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
import pytorch_forecasting
import pytorch_lightning

print("torch:", torch.__version__)
print("forecasting:", pytorch_forecasting.__version__)
print("lightning:", pytorch_lightning.__version__)
print("GPU:", torch.cuda.is_available())

C:\Users\Miguel\OneDrive\Desktop\Libellula\notebooks\Predictive_TFT_model\tft_env\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


torch: 2.11.0+cpu
forecasting: 1.6.1
lightning: 2.6.1
GPU: False


In [1]:
import pandas as pd
import numpy as np
import torch

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
'''
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import LearningRateMonitor'''

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.callbacks import LearningRateMonitor

import matplotlib.pyplot as plt

C:\Users\Miguel\OneDrive\Desktop\Libellula\notebooks\Predictive_TFT_model\tft_env\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


## 2. Load datasets

In [2]:
DATA_PATH = "../../data/processed/financial_tools_datset.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,Date,Price,Open,High,Low,Change %,short_mavg,long_mavg,signal,EMA10,...,MOM30,RSI10,RSI30,RSI200,%K10,%D10,%K30,%D30,%K200,%D200
0,02/11/2026,1.1871,1.1895,1.1928,1.1833,-0.20%,1.187100,1.187100,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02/10/2026,1.1895,1.1913,1.1929,1.1887,-0.17%,1.188300,1.188300,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,02/09/2026,1.1915,1.1817,1.1927,1.1809,0.74%,1.189367,1.189367,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,02/08/2026,1.1827,1.1814,1.1828,1.1810,0.08%,1.187700,1.187700,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02/06/2026,1.1817,1.1778,1.1827,1.1765,0.34%,1.186500,1.186500,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
print(df.shape)
print(df.columns)
df.info()

(1568, 25)
Index(['Date', 'Price', 'Open', 'High', 'Low', 'Change %', 'short_mavg',
       'long_mavg', 'signal', 'EMA10', 'EMA30', 'EMA200', 'ROC10', 'ROC30',
       'MOM10', 'MOM30', 'RSI10', 'RSI30', 'RSI200', '%K10', '%D10', '%K30',
       '%D30', '%K200', '%D200'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1568 entries, 0 to 1567
Data columns (total 25 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Date        1568 non-null   object 
 1   Price       1568 non-null   float64
 2   Open        1568 non-null   float64
 3   High        1568 non-null   float64
 4   Low         1568 non-null   float64
 5   Change %    1568 non-null   object 
 6   short_mavg  1568 non-null   float64
 7   long_mavg   1568 non-null   float64
 8   signal      1568 non-null   float64
 9   EMA10       1559 non-null   float64
 10  EMA30       1539 non-null   float64
 11  EMA200      1369 non-null   float64
 12  ROC10       1559 non-

## 3. Cleaning dataset

In [4]:
# inverter ordem temporal
df = df.iloc[::-1].reset_index(drop=True)

# converter data
df["Date"] = pd.to_datetime(df["Date"])

# criar índice temporal
df["time_idx"] = np.arange(len(df))

# renomear colunas problemáticas
df = df.rename(columns={
    "%K10":"K10",
    "%D10":"D10",
    "%K30":"K30",
    "%D30":"D30",
    "%K200":"K200",
    "%D200":"D200"
})

# remover Change %
df = df.drop(columns=["Change %"])

print(df.head())
print("colunas:", df.columns)

        Date   Price    Open    High     Low  short_mavg  long_mavg  signal  \
0 2020-02-11  1.0914  1.0911  1.0926  1.0890     1.08362   1.094205     0.0   
1 2020-02-12  1.0871  1.0916  1.0926  1.0865     1.08327   1.094078     0.0   
2 2020-02-13  1.0840  1.0874  1.0890  1.0833     1.08335   1.093950     0.0   
3 2020-02-14  1.0830  1.0841  1.0862  1.0827     1.08493   1.093937     0.0   
4 2020-02-17  1.0834  1.0832  1.0852  1.0821     1.08688   1.093953     0.0   

      EMA10     EMA30  ...      RSI10      RSI30     RSI200        K10  \
0  1.087593  1.093965  ...  49.784029  47.966067  47.144045  91.946309   
1  1.086747  1.094142  ...  44.199076  46.565326  46.880856  63.087248   
2  1.086668  1.094628  ...  39.859143  45.543707  46.690443  47.727273   
3  1.087261  1.095361  ...  38.469796  45.217100  46.629037  23.144105   
4  1.088208  1.096213  ...  38.792415  45.322206  46.650424  20.577617   

         D10        K30        D30       K200       D200  time_idx  
0  67.58694

## 4. Creating timeseries structure
### 4.1 Creating targets

In [5]:
df["target"] = df["Price"].shift(-1)

df = df.dropna()

print(df.tail())

           Date   Price    Open    High     Low  short_mavg  long_mavg  \
1362 2025-05-01  1.1292  1.1329  1.1342  1.1266     1.12498   1.148767   
1363 2025-05-02  1.1296  1.1288  1.1381  1.1273     1.12395   1.149538   
1364 2025-05-05  1.1315  1.1319  1.1365  1.1292     1.12263   1.150282   
1365 2025-05-06  1.1370  1.1315  1.1382  1.1280     1.12192   1.150738   
1366 2025-05-07  1.1300  1.1370  1.1379  1.1292     1.12105   1.151032   

      signal     EMA10     EMA30  ...      RSI30     RSI200        K10  \
1362     0.0  1.127866  1.133291  ...  45.049031  46.158490  71.608833   
1363     0.0  1.127569  1.133573  ...  45.179361  46.183615  72.870662   
1364     0.0  1.127118  1.133847  ...  45.787607  46.302730  78.864353   
1365     0.0  1.126144  1.134009  ...  47.580069  46.649273  96.214511   
1366     0.0  1.123732  1.133803  ...  44.927614  46.138786  74.840764   

            D10        K30        D30       K200       D200  time_idx  target  
1362  74.447950  52.183908  45

### 4.2 Creating series ID
TFT needs an identifier:

In [6]:
df["series"] = "EURUSD"

## 5. Creating TFT dataset

### 5.1 Temporal parameters

In [7]:
max_encoder_length = 64
max_prediction_length = 1

### 5.2 Creating TimeSeriesDataset

In [8]:
training = TimeSeriesDataSet(
    df,
    time_idx="time_idx",
    target="target",
    group_ids=["series"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    time_varying_known_reals=[
        "time_idx"
    ],

    time_varying_unknown_reals=[
        'Price','Open','High','Low',
        'short_mavg','long_mavg','signal',
        'EMA10','EMA30','EMA200',
        'ROC10','ROC30',
        'MOM10','MOM30',
        'RSI10','RSI30','RSI200',
        'K10','D10','K30','D30','K200','D200'
    ]
)

### 5.3 Dataloader

In [9]:
train_dataloader = training.to_dataloader(
    train=True,
    batch_size=64,
    num_workers=0
)

## 6. Create model

In [10]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.001,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=4,
)

C:\Users\Miguel\OneDrive\Desktop\Libellula\notebooks\Predictive_TFT_model\tft_env\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
C:\Users\Miguel\OneDrive\Desktop\Libellula\notebooks\Predictive_TFT_model\tft_env\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


### 6.1 Callbacks

In [11]:
early_stop = EarlyStopping(
    monitor="train_loss",
    patience=5,
    mode="min"
)

lr_logger = LearningRateMonitor()

### 6.2 Trainer

In [12]:
trainer = pl.Trainer(
    max_epochs=30,
    gradient_clip_val=0.1,
    callbacks=[early_stop, lr_logger]
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
C:\Users\Miguel\OneDrive\Desktop\Libellula\notebooks\Predictive_TFT_model\tft_env\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


### 6.3 Training the model

In [13]:
trainer.fit(
    tft,
    train_dataloaders=train_dataloader
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
C:\Users\Miguel\OneDrive\Desktop\Libellula\notebooks\Predictive_TFT_model\tft_env\Lib\site-packages\lightning\pytorch\trainer\configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.

   | Name                               | Type                            | Params | Mode  | FLOPs
--------------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train | 0    
1  | logging_metrics                    | ModuleList                      | 0      | train | 0    
2  | input_embeddings                   | MultiEmbedding                  | 0      | train | 0    
3  | prescalers                         | ModuleDict   

Epoch 1:   0%|                       | 0/20 [00:00<?, ?it/s, v_num=0, train_loss_step=0.00938, train_loss_epoch=0.0137]

C:\Users\Miguel\OneDrive\Desktop\Libellula\notebooks\Predictive_TFT_model\tft_env\Lib\site-packages\lightning\pytorch\loops\training_epoch_loop.py:500: ReduceLROnPlateau conditioned on metric val_loss which is not available but strict is set to `False`. Skipping learning rate update.


Epoch 29: 100%|████████████| 20/20 [00:15<00:00,  1.27it/s, v_num=0, train_loss_step=0.00214, train_loss_epoch=0.00211]

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|████████████| 20/20 [00:16<00:00,  1.25it/s, v_num=0, train_loss_step=0.00214, train_loss_epoch=0.00211]


In [14]:
metrics = pd.DataFrame(trainer.callback_metrics, index=[0])

print(metrics)

          lr-Adam      train_loss train_loss_step train_loss_epoch
0  tensor(0.0010)  tensor(0.0021)  tensor(0.0021)   tensor(0.0021)


In [15]:
raw_predictions, x = tft.predict(train_dataloader, mode="raw", return_x=True)

print(raw_predictions)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
C:\Users\Miguel\OneDrive\Desktop\Libellula\notebooks\Predictive_TFT_model\tft_env\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\Users\Miguel\OneDrive\Desktop\Libellula\notebooks\Predictive_TFT_model\tft_env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:485: Your `predict_dataloader`'s sampler has shuffling enabled, it is stron

ValueError: too many values to unpack (expected 2)